
# 08 — Safe Coding Agent Tools: Files, Search, Edits and Tests

**Learning reference:** [BrightPool — Prompt Engineering Course](https://github.com/BrightPool/udemy-prompt-engineering-course)

**Inspired by course notebooks:**
- `basic_chat.ipynb`
- `file_reader.ipynb`
- `code_search.ipynb`
- `file_editor.ipynb`
- `command_runner.ipynb`

## Goal

Build a small, self-contained Python project and implement
tools commonly used by AI coding assistants.

The notebook covers:

- Reading project files.
- Searching source code.
- Editing an approved file.
- Running a predefined test command.
- Validating and restricting tool operations.
- Connecting read-only tools to Gemini for live diagnosis.

**Implementation:** A temporary local Python project
with a controlled tool dispatcher and an optional
Gemini-powered diagnostic workflow.

**API usage:** Offline demonstrations use zero requests.
The optional live section uses at most two Gemini requests.

No external repository or private files are required.



## 1. Coding Agent Tools and Safety

A coding assistant can use tools to interact with
a software project.

Four common operations are:

1. **Read:** Inspect source files and understand existing code.
2. **Search:** Find relevant functions, names, or text.
3. **Edit:** Apply a specific change to a file.
4. **Test:** Run tests to verify the change.

These operations have different risks.

Reading a file does not modify it, while editing files
or running arbitrary commands can have unintended effects.

For this exercise:

- All files are created inside a temporary project directory.
- File paths must remain inside that directory.
- Edits are permitted only in `app.py`.
- Only the predefined unit-test command can execute.
- Arbitrary shell commands are not supported.

These are safeguards for a learning demonstration,
not a complete production security sandbox.


In [1]:

from pathlib import Path
from tempfile import TemporaryDirectory
import subprocess
import sys
import json

workspace = TemporaryDirectory(
    prefix="coding_agent_demo_"
)

project_root = Path(workspace.name).resolve()

(project_root / "tests").mkdir()

(project_root / "app.py").write_text(
    "def add(a, b):\n"
    "    return a - b\n",
    encoding="utf-8"
)

(project_root / "tests" / "test_app.py").write_text(
    "import unittest\n"
    "from app import add\n\n"
    "class TestAdd(unittest.TestCase):\n"
    "    def test_positive_numbers(self):\n"
    "        self.assertEqual(add(2, 3), 5)\n\n"
    "    def test_negative_numbers(self):\n"
    "        self.assertEqual(add(-2, 3), 1)\n\n"
    "if __name__ == '__main__':\n"
    "    unittest.main()\n",
    encoding="utf-8"
)

(project_root / "README.md").write_text(
    "# Sample Python Project\n\n"
    "A small project used to demonstrate "
    "safe coding-agent tools.\n",
    encoding="utf-8"
)

print("Temporary project created.")
print("Files:")

for path in sorted(project_root.rglob("*")):
    if path.is_file():
        print("-", path.relative_to(project_root))


Temporary project created.
Files:
- README.md
- app.py
- tests/test_app.py



## 2. Safe File Reading and Code Search

The first tools are read-only.

`read_file` opens a project file and returns its content.

`search_code` searches approved project files for a
literal text pattern and returns matching lines.

A path-validation helper checks that the resolved path
remains within our temporary project directory.

These tools do not make API requests.


In [2]:

ALLOWED_EXTENSIONS = {".py", ".md"}
MAX_FILE_BYTES = 20_000


def safe_project_path(relative_path):
    if not isinstance(relative_path, str):
        raise ValueError("Path must be a string.")

    requested = Path(relative_path)

    if requested.is_absolute():
        raise ValueError("Absolute paths are not allowed.")

    resolved = (project_root / requested).resolve()

    try:
        resolved.relative_to(project_root)
    except ValueError:
        raise ValueError(
            "Path must remain inside the project."
        )

    if not resolved.is_file():
        raise FileNotFoundError(relative_path)

    if resolved.suffix not in ALLOWED_EXTENSIONS:
        raise ValueError("File type not allowed.")

    if resolved.stat().st_size > MAX_FILE_BYTES:
        raise ValueError("File is too large.")

    return resolved


def read_file(path):
    file_path = safe_project_path(path)
    return file_path.read_text(encoding="utf-8")


def search_code(query):
    if not isinstance(query, str) or not query:
        raise ValueError("Search query must not be empty.")

    matches = []

    for file_path in sorted(project_root.rglob("*")):
        if not file_path.is_file():
            continue

        if file_path.suffix not in ALLOWED_EXTENSIONS:
            continue

        if "__pycache__" in file_path.parts:
            continue

        relative_path = str(
            file_path.relative_to(project_root)
        )

        # Apply the same path and size checks.
        text = read_file(relative_path)

        for line_number, line in enumerate(
            text.splitlines(), start=1
        ):
            if query in line:
                matches.append({
                    "file": relative_path,
                    "line": line_number,
                    "text": line.strip()
                })

    return matches


print("APP.PY CONTENT:\n")
print(read_file("app.py"))

print("\nSEARCH RESULTS:")
print(search_code("def add"))


APP.PY CONTENT:

def add(a, b):
    return a - b


SEARCH RESULTS:
[{'file': 'app.py', 'line': 1, 'text': 'def add(a, b):'}]



## 3. Restricted Editing and Test Execution

Next, implement two tools that can affect the project:

**Edit tool**

Replace one exact piece of text in `app.py`.
The edit must have exactly one matching location.

**Test tool**

Run Python's built-in `unittest` test discovery.

The tool does not accept a shell-command string.
It always executes the same predefined test operation.

This reduces the capabilities available to callers.


In [3]:

def edit_file(path, old_text, new_text):
    if path != "app.py":
        raise PermissionError(
            "Only app.py can be edited."
        )

    if not isinstance(old_text, str) or not old_text:
        raise ValueError("old_text must not be empty.")

    if not isinstance(new_text, str):
        raise ValueError("new_text must be a string.")

    file_path = safe_project_path(path)
    original = file_path.read_text(encoding="utf-8")

    occurrences = original.count(old_text)

    if occurrences != 1:
        raise ValueError(
            "Expected exactly one matching occurrence; "
            f"found {occurrences}."
        )

    updated = original.replace(old_text, new_text, 1)

    if len(updated.encode("utf-8")) > MAX_FILE_BYTES:
        raise ValueError("Edited file would be too large.")

    file_path.write_text(updated, encoding="utf-8")

    return {
        "file": path,
        "changed": True,
        "replacements": 1
    }


def run_tests():
    completed = subprocess.run(
        [
            sys.executable,
            "-m",
            "unittest",
            "discover",
            "-s",
            "tests",
            "-v"
        ],
        cwd=project_root,
        capture_output=True,
        text=True,
        timeout=15,
        check=False
    )

    return {
        "returncode": completed.returncode,
        "passed": completed.returncode == 0,
        "output": (
            completed.stdout + completed.stderr
        )[-5000:]
    }


print("Restricted editing and testing tools ready.")


Restricted editing and testing tools ready.


In [4]:

before_fix = run_tests()

print("Tests passed:", before_fix["passed"])
print("Return code:", before_fix["returncode"])
print("\nTEST OUTPUT:\n")
print(before_fix["output"])

assert before_fix["passed"] is False

print("\nFailure detected as expected.")


Tests passed: False
Return code: 1

TEST OUTPUT:

test_negative_numbers (test_app.TestAdd.test_negative_numbers) ... FAIL
test_positive_numbers (test_app.TestAdd.test_positive_numbers) ... FAIL

FAIL: test_negative_numbers (test_app.TestAdd.test_negative_numbers)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/tmp/coding_agent_demo_d_3nuuz6/tests/test_app.py", line 9, in test_negative_numbers
    self.assertEqual(add(-2, 3), 1)
    ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^
AssertionError: -5 != 1

FAIL: test_positive_numbers (test_app.TestAdd.test_positive_numbers)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/tmp/coding_agent_demo_d_3nuuz6/tests/test_app.py", line 6, in test_positive_numbers
    self.assertEqual(add(2, 3), 5)
    ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^
AssertionError: -1 != 5

----------------------------------------------------------------------
Ran 2 

In [5]:

edit_result = edit_file(
    path="app.py",
    old_text="return a - b",
    new_text="return a + b"
)

print("EDIT RESULT:")
print(edit_result)

print("\nUPDATED FILE:\n")
print(read_file("app.py"))

after_fix = run_tests()

print("\nTESTS PASSED:", after_fix["passed"])
print(after_fix["output"])

assert after_fix["passed"] is True


EDIT RESULT:
{'file': 'app.py', 'changed': True, 'replacements': 1}

UPDATED FILE:

def add(a, b):
    return a + b


TESTS PASSED: True
test_negative_numbers (test_app.TestAdd.test_negative_numbers) ... ok
test_positive_numbers (test_app.TestAdd.test_positive_numbers) ... ok

----------------------------------------------------------------------
Ran 2 tests in 0.000s

OK




## 4. Tool Dispatcher and Safety Checks

A coding assistant needs a way to choose between
available tools.

The dispatcher maps tool names to approved Python
functions.

This prevents callers from requesting arbitrary
Python functions by name.

We also verify that:

- Paths outside the project are rejected.
- Unauthorized file edits are rejected.
- Unknown tools are rejected.
- The repaired application passes its tests.


In [6]:

CODING_TOOLS = {
    "read_file": read_file,
    "search_code": search_code,
    "edit_file": edit_file,
    "run_tests": run_tests
}


def dispatch_coding_tool(name, arguments):
    if name not in CODING_TOOLS:
        raise ValueError(
            f"Unknown tool: {name}"
        )

    if not isinstance(arguments, dict):
        raise ValueError(
            "Tool arguments must be an object."
        )

    return CODING_TOOLS[name](**arguments)


# Verify a working lookup.
assert "return a + b" in dispatch_coding_tool(
    "read_file",
    {"path": "app.py"}
)

# Unknown tools must be rejected.
try:
    dispatch_coding_tool(
        "execute_arbitrary_code", {}
    )
    raise AssertionError("Unknown tool was accepted.")
except ValueError:
    pass

# Paths outside the project must be rejected.
try:
    read_file("../outside.py")
    raise AssertionError("Unsafe path was accepted.")
except (ValueError, FileNotFoundError):
    pass

# Editing the tests is not permitted.
try:
    edit_file(
        "tests/test_app.py",
        "old",
        "new"
    )
    raise AssertionError(
        "Unauthorized edit was accepted."
    )
except PermissionError:
    pass

assert dispatch_coding_tool(
    "run_tests", {}
)["passed"] is True

print("All coding-tool safety checks passed.")
print("Gemini API requests: 0")


All coding-tool safety checks passed.
Gemini API requests: 0



## 5. Live Gemini Coding Assistant

Connect Gemini to the safe project-inspection tools.

The model can request:

- `read_file` to inspect source code.
- `search_code` to locate code.
- `run_tests` to check the application.

The model cannot call `edit_file` in this live section.

The question asks Gemini to inspect the implementation
and determine whether the tests pass.

**Request budget:** At most two Gemini requests.

1. Select diagnostic tools.
2. Generate a final answer using their results.


Only one model-driven tool-selection round is implemented.

If Gemini does not request a required diagnostic operation,
Python performs that operation using the restricted tools.

The final response is generated from the complete
diagnostic evidence, without requiring another
tool-selection round.



In [7]:

from openai import OpenAI
from getpass import getpass

MODEL = "gemini-3.5-flash-lite"

gemini_key = getpass("Enter your Gemini API key: ")

client = OpenAI(
    api_key=gemini_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    max_retries=0,
    timeout=120.0
)

safe_client = client.with_options(
    max_retries=0,
    timeout=120.0
)

print("Selected model:", MODEL)
print("Client ready. No API request made.")


Enter your Gemini API key: ··········
Selected model: gemini-3.5-flash-lite
Client ready. No API request made.


In [8]:

gemini_coding_tools = [
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": (
                "Read the contents of an approved "
                "file inside the temporary project."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {
                        "type": "string",
                        "description": (
                            "Project-relative path, "
                            "for example app.py"
                        )
                    }
                },
                "required": ["path"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_code",
            "description": (
                "Search project source files for "
                "a literal text pattern."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Text to search for"
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "run_tests",
            "description": (
                "Run the predefined Python unittest "
                "suite and return the test results."
            ),
            "parameters": {
                "type": "object",
                "properties": {}
            }
        }
    }
]

# Restrict the live dispatcher to read-only
# diagnostics and the predefined test command.
LIVE_TOOLS = {
    "read_file": read_file,
    "search_code": search_code,
    "run_tests": run_tests
}


def dispatch_live_tool(name, arguments):
    if name not in LIVE_TOOLS:
        raise ValueError(
            f"Tool not available to Gemini: {name}"
        )

    if not isinstance(arguments, dict):
        raise ValueError(
            "Tool arguments must be an object."
        )

    return LIVE_TOOLS[name](**arguments)


print("Gemini tools:", list(LIVE_TOOLS))


Gemini tools: ['read_file', 'search_code', 'run_tests']


In [10]:

# Live coding assistant with application-enforced verification.
# Maximum: 2 Gemini API requests.

conversation = [
    {
        "role": "system",
        "content": (
            "You are a careful Python coding assistant. "
            "Use available tools to inspect code and tests. "
            "Base your conclusions only on retrieved evidence. "
            "Do not claim you edited any files."
        )
    },
    {
        "role": "user",
        "content": (
            "Inspect app.py and run the unit tests. "
            "Explain what the add function does, "
            "whether the tests pass, and the evidence."
        )
    }
]

# Request 1: Gemini chooses diagnostic tools.
first = safe_client.chat.completions.create(
    model=MODEL,
    messages=conversation,
    tools=gemini_coding_tools,
    tool_choice="required"
)

assistant_message = first.choices[0].message
calls = assistant_message.tool_calls or []

print("Gemini requested:", [
    call.function.name for call in calls
])

# Preserve the model's tool-call message.
if calls:
    conversation.append(
        assistant_message.model_dump(exclude_none=True)
    )

model_results = {}

for call in calls:
    name = call.function.name

    try:
        arguments = json.loads(call.function.arguments)

        result = dispatch_live_tool(name, arguments)

        # Save successful results and their arguments.
        model_results[(name, json.dumps(
            arguments, sort_keys=True
        ))] = result

    except Exception as error:
        result = {
            "error": type(error).__name__,
            "message": str(error)
        }

    print("\nModel-selected tool:", name)
    print("Result:", result)

    conversation.append({
        "role": "tool",
        "tool_call_id": call.id,
        "content": json.dumps(result)
    })


# Application-enforced diagnostics.
# Reuse valid model-selected results when possible.

read_key = (
    "read_file",
    json.dumps({"path": "app.py"}, sort_keys=True)
)

test_key = (
    "run_tests",
    json.dumps({}, sort_keys=True)
)

if read_key in model_results:
    code_result = model_results[read_key]
    print("\nRequired file inspection: already completed.")
else:
    code_result = read_file("app.py")
    print("\nRequired file inspection: enforced by Python.")

if test_key in model_results:
    test_result = model_results[test_key]
    print("Required tests: already completed.")
else:
    test_result = run_tests()
    print("Required tests: enforced by Python.")

print("\nVerified test status:", test_result["passed"])

# Give Gemini the complete, verified evidence.
# These application-run results are not falsely
# presented as model-requested tool calls.

conversation.append({
    "role": "user",
    "content": (
        "The application has completed its mandatory "
        "diagnostic checks. Use these verified results "
        "to answer the original question.\n\n"
        "APP.PY:\n"
        f"{code_result}\n\n"
        "UNIT TEST RESULTS:\n"
        f"{json.dumps(test_result)}\n\n"
        "Explain the function and test outcome. "
        "Distinguish observations from assumptions."
    )
})

# Request 2: Gemini produces an evidence-based answer.
second = safe_client.chat.completions.create(
    model=MODEL,
    messages=conversation
)

final_answer = second.choices[0].message.content

if not final_answer:
    raise ValueError("Gemini returned an empty answer.")

print("\nGEMINI FINAL ANSWER:\n")
print(final_answer)


Gemini requested: ['read_file']

Model-selected tool: read_file
Result: def add(a, b):
    return a + b


Required file inspection: already completed.
Required tests: enforced by Python.

Verified test status: True

GEMINI FINAL ANSWER:

Based on the provided diagnostic results and source code, here is the explanation:

### 1. What the `add` function does
* **Observation:** The `add` function takes two parameters (`a` and `b`) and returns their sum using the `+` operator (`return a + b`). 
* **Assumption (Context-dependent):** Depending on the types of inputs passed into `a` and `b`, this function can perform standard arithmetic addition (for numbers/integers/floats) or concatenation (for strings or lists), though the provided unit tests explicitly focus on numerical addition.

### 2. Whether the tests pass
* **Observation:** Yes, the unit tests pass successfully. 
* **Evidence:** The unit test execution results show a `returncode` of `0`, a `passed` status of `true`, and the output in


## 6. Verification and Security Boundaries

The offline demonstration verified a complete
read → test → edit → test workflow.

The live demonstration gives Gemini a smaller,
read-only diagnostic toolset.

### Security Boundaries

- Tools are selected from an explicit allowlist.
- File access is restricted to the temporary project.
- Arbitrary shell commands are not exposed.
- File editing is restricted to `app.py` offline.
- Gemini cannot use the editing tool in the live workflow.
- Test execution is restricted to a predefined command.

### Remaining Risks

- Executing tests still runs Python project code.
- The path restrictions are not a full operating-system sandbox.
- A production coding agent would need stronger process,
  filesystem, network and permission isolation.
- Tool outputs should be treated as untrusted information.
- Human approval should precede consequential changes.

This notebook is an educational demonstration,
not a production-ready autonomous coding agent.



## Findings and Limitations

### Offline Tool Workflow

- Created a temporary Python project containing a deliberate bug.
- Implemented file reading and literal code search.
- Used unit tests to identify incorrect behaviour.
- Applied a restricted text replacement to correct the bug.
- Verified that both unit tests passed after the correction.
- Confirmed that unknown tools, unsafe paths, and unauthorized edits were rejected.

### Live Gemini Workflow

- Exposed approved diagnostic tools to Gemini through function calling.
- Executed model-requested operations using a controlled Python dispatcher.
- Enforced mandatory file inspection and unit testing at the application level.
- Supplied the diagnostic evidence to Gemini for its final explanation.
- Limited the workflow to one model-driven tool-selection round and at most two model requests.

### Observed Live Results

- Gemini 3.5 Flash Lite requested `read_file` to inspect `app.py`.
- Python reused the retrieved file content and independently executed `run_tests`, which Gemini had not requested.
- Both unit tests passed after the earlier code correction.
- Gemini correctly explained the `add` function and supported its conclusion with the test results.
- The live demonstration completed in two Gemini requests.

### Limitations

- The project is a small, synthetic example rather than a real software repository.
- Code search uses literal text matching rather than semantic or syntax-aware search.
- The edit tool supports exact text replacement rather than general code refactoring.
- Only one predefined test command is supported.
- Gemini can inspect the project but cannot edit files in the live workflow.
- Running tests executes project code; the temporary directory and path restrictions do not provide a fully isolated security sandbox.
- The workflow does not implement autonomous multi-step coding or repeated model-driven verification.

### Conclusion

This notebook demonstrates the foundations of a controlled AI-assisted coding workflow: restricted file access, code search, targeted editing, automated testing, and LLM-guided diagnostics.

The offline exercise verifies a complete bug-detection and correction cycle. The live Gemini exercise demonstrates a complementary design in which the model selects diagnostic tools, while Python independently enforces required verification steps.

Together, these examples illustrate how deterministic application controls can improve the reliability of AI-assisted coding without granting the model unrestricted access to files or commands.
